In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Define Hyperparameters
batch_size = 128
num_workers = 2
learning_rate = 1e-3
epochs = 15

In [4]:
# transform
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=5),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [6]:
#Load data
train_set = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)
test_set = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

In [7]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Input = (3,32,32)
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(2), # 32x32 -> 16x16

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2), # 16x16 -> 8x8

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), # 8x8->4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,num_classes)
        )
    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1,

In [9]:
# loss , optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)

In [10]:
# Training loop
def train_one_epoch():
    model.train()
    running_loss, correct, total = 0,0,0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss+=loss
        _, predictions = outputs.max(1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/ len(train_loader), 100. * correct/total

In [13]:
# evaluation loop
def evaluate():
    model.eval()
    correct, total = 0,0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predictions = outputs.max(1)
            correct += predictions.eq(labels).sum().item()
            total += labels.size(0)
    return 100. * correct/total

In [14]:
# training model
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% |"
          f"Test Acc: {test_acc:.2f}%| LR:{current_lr:.6f}")

Epoch 1/15 | Loss: 1.2020 | Train Acc: 56.51% |Test Acc: 62.16%| LR:0.000989
Epoch 2/15 | Loss: 1.0395 | Train Acc: 63.33% |Test Acc: 67.32%| LR:0.000957
Epoch 3/15 | Loss: 0.9398 | Train Acc: 67.09% |Test Acc: 68.20%| LR:0.000905
Epoch 4/15 | Loss: 0.8683 | Train Acc: 69.79% |Test Acc: 73.50%| LR:0.000835
Epoch 5/15 | Loss: 0.8161 | Train Acc: 71.91% |Test Acc: 75.59%| LR:0.000750
Epoch 6/15 | Loss: 0.7693 | Train Acc: 73.39% |Test Acc: 76.07%| LR:0.000655
Epoch 7/15 | Loss: 0.7231 | Train Acc: 75.29% |Test Acc: 77.70%| LR:0.000552
Epoch 8/15 | Loss: 0.6865 | Train Acc: 76.57% |Test Acc: 79.75%| LR:0.000448
Epoch 9/15 | Loss: 0.6508 | Train Acc: 77.81% |Test Acc: 78.28%| LR:0.000345
Epoch 10/15 | Loss: 0.6203 | Train Acc: 78.84% |Test Acc: 79.75%| LR:0.000250
Epoch 11/15 | Loss: 0.5911 | Train Acc: 80.03% |Test Acc: 82.24%| LR:0.000165
Epoch 12/15 | Loss: 0.5633 | Train Acc: 80.83% |Test Acc: 82.77%| LR:0.000095
Epoch 13/15 | Loss: 0.5551 | Train Acc: 81.05% |Test Acc: 82.89%| LR:0.00